# End-to-End Medallion Pipeline Orchestrator

Executes the full Lakehouse pipeline in order:
1. **`01_Config`**: Initializes catalogs, schemas, and volumes.
2. **`00_Data_Generator`**: Generates synthetic e-commerce parquet data into the landing volume.
3. **`02_Bronze_Layer`**: Streams raw data into `bronze_prod.ecommerce.*` via Auto Loader.
4. **`03_Silver_Layer`**: Conforms domains, enforces PII tags, and runs SCD Type 2 MERGE into `silver_prod`.
5. **`04_Gold_Layer`**: Builds Kimball Star Schema, Power BI date table, and business reporting tables in `gold_prod`.


In [0]:
# Orchestrator Configuration Widgets
dbutils.widgets.dropdown("generate_sample_data", "true", ["true", "false"], "Generate Sample Data")
dbutils.widgets.dropdown("batch_mode", "initial", ["initial", "update"], "Data Batch Mode (initial | update)")

gen_data = dbutils.widgets.get("generate_sample_data")
batch_mode = dbutils.widgets.get("batch_mode")


In [0]:
import time
from datetime import datetime

def run_step(notebook_name, args=None):
    print(f"\n{'='*70}")
    print(f"[ORCHESTRATOR] Running: {notebook_name} at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*70}")
    t0 = time.time()
    result = dbutils.notebook.run(notebook_name, 3600, args or {})
    elapsed = time.time() - t0
    print(f"✓ Completed: {notebook_name} in {elapsed:.2f} seconds.")
    return notebook_name, elapsed

execution_log = []

# Step 1: Environment & Config
execution_log.append(run_step("01_Config"))

# Step 2: Data Generator (optional)
if gen_data == "true":
    execution_log.append(run_step("00_Data_Generator", {"batch_mode": batch_mode}))

# Step 3: Bronze Auto Loader Ingestion
execution_log.append(run_step("02_Bronze_Layer", {"p_dataset": "all"}))

# Step 4: Silver Conformed & SCD Type 2
execution_log.append(run_step("03_Silver_Layer"))

# Step 5: Gold Star Schema & Business Marts
execution_log.append(run_step("04_Gold_Layer"))

print("\n" + "="*70)
print("[ORCHESTRATOR] Pipeline Execution Summary:")
print("="*70)
for nb, duration in execution_log:
    print(f"  • {nb:<25} : {duration:.2f} s")
total_time = sum(d for _, d in execution_log)
print(f"\nTotal Pipeline Execution Time: {total_time:.2f} s ({total_time/60:.2f} min)")
